# REE Distribution Data Engineering - Qiwei Liang

 
**Scope:** Locality distribution of 17 rare-earth elements in the top 5 countries by rare-earth reserves.  

**Hypothesis1**:

Do global REE occurrences form distinct spatial clusters that correlate with specific tectonic plate boundaries?

**Hypothesis 2**:

Is there a significant difference in the proportion of light rare earth and heavy rare earth minerals among different clustering groups?

**Important note:**

1.Mindat locality data shows where elements are reported. It does not provide reliable ore grade, concentration percentage, or reserve tonnage by element. This notebook analyzes locality distribution, not chemical concentration.

2.The top 5 country reserve numbers are from USGS Mineral Commodity Summaries 2026, Rare Earths chapter.
https://pubs.usgs.gov/periodicals/mcs2025/mcs2025-rare-earths.pdf?utm

## Scope

This notebook focuses on the requirement: top 5 rare-earth reserve countries and 17 REE elements.

**Countries:** China, Brazil, Australia, Russia, Vietnam  
**Elements:** Sc, Y, La, Ce, Pr, Nd, Pm, Sm, Eu, Gd, Tb, Dy, Ho, Er, Tm, Yb, Lu




In [ ]:
import json
import time
from pathlib import Path

import pandas as pd
import requests

# Store raw API responses separately from cleaned outputs.
RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Mindat API key used by every request below.
API_KEY = "3b54da79accb60ccb08008b7271d668c"
HEADERS = {
    "Authorization": f"Token {API_KEY}",
    "Accept": "application/json",
    "User-Agent": "Qiwei-REE-project",
}

# High page size reduces pagination while staying simple for a class project notebook.
PAGE_SIZE = 500
SLEEP_SECONDS = 0.15

# USGS 2026 rare-earth reserves, in metric tons REO equivalent.
TOP_REE_COUNTRIES = pd.DataFrame([
    {"reserve_rank": 1, "country": "China", "reserves_reo_tonnes": 44_000_000},
    {"reserve_rank": 2, "country": "Brazil", "reserves_reo_tonnes": 21_000_000},
    {"reserve_rank": 3, "country": "Australia", "reserves_reo_tonnes": 6_300_000},
    {"reserve_rank": 4, "country": "Russia", "reserves_reo_tonnes": 3_800_000},
    {"reserve_rank": 5, "country": "Vietnam", "reserves_reo_tonnes": 3_500_000},
])

# Project definition of 17 REE elements: Sc + Y + 15 lanthanides.
REE_ELEMENTS = pd.DataFrame([
    {"symbol": "Sc", "element_name": "Scandium", "ree_group": "scandium"},
    {"symbol": "Y", "element_name": "Yttrium", "ree_group": "hree"},
    {"symbol": "La", "element_name": "Lanthanum", "ree_group": "lree"},
    {"symbol": "Ce", "element_name": "Cerium", "ree_group": "lree"},
    {"symbol": "Pr", "element_name": "Praseodymium", "ree_group": "lree"},
    {"symbol": "Nd", "element_name": "Neodymium", "ree_group": "lree"},
    {"symbol": "Pm", "element_name": "Promethium", "ree_group": "lree"},
    {"symbol": "Sm", "element_name": "Samarium", "ree_group": "lree"},
    {"symbol": "Eu", "element_name": "Europium", "ree_group": "lree"},
    {"symbol": "Gd", "element_name": "Gadolinium", "ree_group": "hree"},
    {"symbol": "Tb", "element_name": "Terbium", "ree_group": "hree"},
    {"symbol": "Dy", "element_name": "Dysprosium", "ree_group": "hree"},
    {"symbol": "Ho", "element_name": "Holmium", "ree_group": "hree"},
    {"symbol": "Er", "element_name": "Erbium", "ree_group": "hree"},
    {"symbol": "Tm", "element_name": "Thulium", "ree_group": "hree"},
    {"symbol": "Yb", "element_name": "Ytterbium", "ree_group": "hree"},
    {"symbol": "Lu", "element_name": "Lutetium", "ree_group": "hree"},
])

print("API key used:", API_KEY[:6] + "..." + API_KEY[-4:])
print("Countries:", TOP_REE_COUNTRIES["country"].tolist())
print("Elements:", REE_ELEMENTS["symbol"].tolist())


API key used: 3b54da...668c
Countries: ['China', 'Brazil', 'Australia', 'Russia', 'Vietnam']
Elements: ['Sc', 'Y', 'La', 'Ce', 'Pr', 'Nd', 'Pm', 'Sm', 'Eu', 'Gd', 'Tb', 'Dy', 'Ho', 'Er', 'Tm', 'Yb', 'Lu']


## Pull Data From Mindat

For each country and each REE element, the notebook pulls Mindat localities where that element is reported.



In [ ]:
def save_json(data, path):
     # Save data as a UTF-8 JSON file
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")


def fetch_localities(country, element):
    #Fetch Mindat localities for one country and one REE element.
    url = "https://api.mindat.org/v1/localities/"
    params = {
        "country": country,
        "elements_inc": element,
        "fields": "id,txt,country,latitude,longitude,elements,datemodify,parent,coordsystem",
        "page-size": PAGE_SIZE,
    }

    rows = []
    page = 1

    while url:
        response = requests.get(url, headers=HEADERS, params=params if page == 1 else None, timeout=60)

        # A 400 response can happen for unsupported or empty element filters; keep the run moving.
        if response.status_code == 400:
            print(f"{country} / {element}: API returned 400, saved as 0 rows")
            return []

        if response.status_code == 401:
            raise RuntimeError("Mindat API key was rejected. Check API_KEY.")

        response.raise_for_status()
        data = response.json()

        # Attach query metadata so each row keeps its country-element context after merging.
        for row in data.get("results", []):
            row["query_country"] = country
            row["query_element"] = element
            rows.append(row)

        url = data.get("next")
        page += 1
        time.sleep(SLEEP_SECONDS)

    return rows


In [ ]:
raw_rows = []

# Run 5 countries x 17 elements = 85 API queries.
for country in TOP_REE_COUNTRIES["country"]:
    for element in REE_ELEMENTS["symbol"]:
        rows = fetch_localities(country, element)
        raw_rows.extend(rows)
        print(f"{country} / {element}: {len(rows)} localities")

raw_path = RAW_DIR / "mindat_top5_country_17ree_localities_raw.json"
save_json(raw_rows, raw_path)

raw_df = pd.DataFrame(raw_rows)
print("Raw rows:", len(raw_df))
raw_df.head()


China / Sc: 30 localities
China / Y: 405 localities
China / La: 141 localities
China / Ce: 359 localities
China / Pr: 0 localities
China / Nd: 208 localities
China / Pm: 0 localities
China / Sm: 0 localities
China / Eu: 0 localities
China / Gd: 0 localities
China / Tb: 0 localities
China / Dy: 0 localities
China / Ho: 0 localities
China / Er: 0 localities
China / Tm: 0 localities
China / Yb: 14 localities
China / Lu: 0 localities
Brazil / Sc: 2 localities
Brazil / Y: 104 localities
Brazil / La: 41 localities
Brazil / Ce: 132 localities
Brazil / Pr: 0 localities
Brazil / Nd: 44 localities
Brazil / Pm: 0 localities
Brazil / Sm: 0 localities
Brazil / Eu: 0 localities
Brazil / Gd: 0 localities
Brazil / Tb: 0 localities
Brazil / Dy: 0 localities
Brazil / Ho: 0 localities
Brazil / Er: 0 localities
Brazil / Tm: 0 localities
Brazil / Yb: 0 localities
Brazil / Lu: 0 localities
Australia / Sc: 25 localities
Australia / Y: 283 localities
Australia / La: 111 localities
Australia / Ce: 267 localiti

,id,txt,latitude,longitude,datemodify,elements,country,coordsystem,parent,query_country,query_element
0,693,China,0.000000,0.000000,2024-11-26 07:44:08,-Si-O-Ag-S-Fe-Se-Ca-Mg-H-C-As-Zn-Al-K-Na-Ti-Ce...,China,0,0,China,Sc
1,701,"Guizhou, China",0.000000,0.000000,2025-09-20 18:13:10,-Ag-S-Ca-Fe-Mg-Si-O-H-As-Cu-Hg-Mn-Al-Na-Ti-Pb-...,China,0,693,China,Sc
2,705,"Hunan, China",0.000000,0.000000,2024-10-23 13:43:34,-Ag-S-Ca-Fe-Mg-Si-O-H-Al-K-Ce-Nb-Ti-Bi-Cu-Pb-M...,China,0,693,China,Sc
3,706,"Chenzhou, Hunan, China",0.000000,0.000000,2022-06-16 15:13:28,-Ag-S-Ca-Fe-Mg-Si-O-H-Bi-Cu-Pb-Mn-Al-Na-Te-K-T...,China,0,705,China,Sc
4,715,"Xianghualing Mine, Xianghualing Sn-polymetalli...",25.458611,112.567222,2026-01-25 16:21:47,-Ag-S-Ca-Fe-Mg-Si-O-H-Al-Na-K-As-Ba-Li-Be-Ti-F...,China,0,157020,China,Sc


## Clean Data

This section cleans raw Mindat locality data and builds the final tables for later analysis.


In [ ]:
def parse_elements(value):
    """Convert a Mindat element string like '-La-Ce-Nd-' into a Python list."""
    if pd.isna(value):
        return []
    text = str(value).strip()
    if text.startswith("-") and text.endswith("-"):
        return [x for x in text.split("-") if x]
    return [x.strip() for x in text.split(",") if x.strip()]


clean_columns = [
    "reserve_rank", "query_country", "reserves_reo_tonnes", "query_element", "element_name", "ree_group",
    "locality_id", "locality_name", "country", "latitude", "longitude", "analysis_ready",
    "reported_elements", "datemodify", "parent", "coordsystem", "mindat_locality_url",
]

if raw_df.empty:
    clean = pd.DataFrame(columns=clean_columns)
else:
    clean = raw_df.rename(columns={"id": "locality_id", "txt": "locality_name"}).copy()
    clean["latitude"] = pd.to_numeric(clean["latitude"], errors="coerce")
    clean["longitude"] = pd.to_numeric(clean["longitude"], errors="coerce")
    clean["reported_elements"] = clean["elements"].map(parse_elements)

    # Keep rows with real map coordinates and remove the common placeholder coordinate (0, 0).
    valid_lat = clean["latitude"].between(-90, 90)
    valid_lon = clean["longitude"].between(-180, 180)
    not_zero = ~((clean["latitude"].fillna(0) == 0) & (clean["longitude"].fillna(0) == 0))
    clean["analysis_ready"] = valid_lat & valid_lon & not_zero

    clean["mindat_locality_url"] = clean["locality_id"].map(lambda x: f"https://www.mindat.org/loc-{int(x)}.html")

    # Add reserve-country metadata and element-group metadata to every locality row.
    country_info = TOP_REE_COUNTRIES.rename(columns={"country": "query_country"})
    element_info = REE_ELEMENTS.rename(columns={"symbol": "query_element"})
    clean = clean.merge(country_info, on="query_country", how="left")
    clean = clean.merge(element_info, on="query_element", how="left")
    clean = clean.drop_duplicates(subset=["query_country", "query_element", "locality_id"])
    clean = clean[[c for c in clean_columns if c in clean.columns]]

if clean.empty:
    analysis_ready = clean.copy()
else:
    analysis_ready = clean[clean["analysis_ready"]].copy()

# Long summary table: one row per country-element pair.
summary = (
    analysis_ready
    .groupby(["reserve_rank", "query_country", "reserves_reo_tonnes", "query_element", "element_name", "ree_group"], dropna=False)
    .agg(locality_count=("locality_id", "nunique"))
    .reset_index()
    .sort_values(["reserve_rank", "query_element"])
)

# Wide matrix table: countries as rows, elements as columns, locality counts as values.
matrix = summary.pivot_table(
    index="query_country",
    columns="query_element",
    values="locality_count",
    fill_value=0,
    aggfunc="sum",
)
matrix = matrix.reindex(index=TOP_REE_COUNTRIES["country"], columns=REE_ELEMENTS["symbol"], fill_value=0)

print("Clean rows:", len(clean))
print("Analysis-ready rows:", len(analysis_ready))
matrix


Clean rows: 3546
Analysis-ready rows: 1338


symbol,Sc,Y,La,Ce,Pr,Nd,Pm,Sm,Eu,Gd,Tb,Dy,Ho,Er,Tm,Yb,Lu
country,,,,,,,,,,,,,,,,,
China,6,114,40,109,0,63,0,0,0,0,0,0,0,0,0,3,0
Brazil,0,52,17,67,0,21,0,0,0,0,0,0,0,0,0,0,0
Australia,7,150,42,111,0,38,0,0,0,0,0,0,0,0,0,0,0
Russia,8,125,62,216,0,59,0,4,0,1,0,0,0,0,0,2,0
Vietnam,0,4,4,7,0,6,0,0,0,0,0,0,0,0,0,0,0


## Save Outputs

These are Qiwei's final data-engineering deliverables.


In [ ]:
clean_path = PROCESSED_DIR / "top5_17ree_localities_clean.csv"
analysis_path = PROCESSED_DIR / "top5_17ree_localities_analysis_ready.csv"
summary_path = PROCESSED_DIR / "top5_17ree_distribution_summary.csv"
matrix_path = PROCESSED_DIR / "top5_17ree_distribution_matrix.csv"
country_path = PROCESSED_DIR / "top5_ree_reserve_countries.csv"
elements_path = PROCESSED_DIR / "ree_17_elements.csv"
quality_path = PROCESSED_DIR / "top5_17ree_data_quality_report.json"

clean.to_csv(clean_path, index=False)
analysis_ready.to_csv(analysis_path, index=False)
summary.to_csv(summary_path, index=False)
matrix.to_csv(matrix_path)
TOP_REE_COUNTRIES.to_csv(country_path, index=False)
REE_ELEMENTS.to_csv(elements_path, index=False)

quality_report = {
    "scope": "Top 5 rare-earth reserve countries and 17 REE element locality distribution from Mindat.",
    "api_key_used_preview": API_KEY[:6] + "..." + API_KEY[-4:],
    "country_count": len(TOP_REE_COUNTRIES),
    "element_count": len(REE_ELEMENTS),
    "raw_rows": len(raw_df),
    "clean_rows": len(clean),
    "analysis_ready_rows": len(analysis_ready),
    "unique_localities": int(analysis_ready["locality_id"].nunique()) if not analysis_ready.empty else 0,
    "outputs": {
        "clean": str(clean_path),
        "analysis_ready": str(analysis_path),
        "summary": str(summary_path),
        "matrix": str(matrix_path),
        "countries": str(country_path),
        "elements": str(elements_path),
    },
}
save_json(quality_report, quality_path)

quality_report


{'scope': 'Top 5 rare-earth reserve countries and 17 REE element locality distribution from Mindat.',
 'api_key_used_preview': '3b54da...668c',
 'country_count': 5,
 'element_count': 17,
 'raw_rows': 3546,
 'clean_rows': 3546,
 'analysis_ready_rows': 1338,
 'unique_localities': 714,
 'outputs': {'clean': 'data\\processed\\top5_17ree_localities_clean.csv',
  'analysis_ready': 'data\\processed\\top5_17ree_localities_analysis_ready.csv',
  'summary': 'data\\processed\\top5_17ree_distribution_summary.csv',
  'matrix': 'data\\processed\\top5_17ree_distribution_matrix.csv',
  'countries': 'data\\processed\\top5_ree_reserve_countries.csv',
  'elements': 'data\\processed\\ree_17_elements.csv'}}

## Quick Checks

Basic checks before handing files to teammates.


In [ ]:
#Replace API_KEY with your own Mindat API key before running.
#You can apply for a key in minat.org, it will take a few days to get approved.
#But it's free for academic use and will allow you to run the notebook without hitting authentication errors.
assert API_KEY == "PASTE_YOUR_MINDAT_API_KEY_HERE"

assert len(TOP_REE_COUNTRIES) == 5
assert len(REE_ELEMENTS) == 17
assert set(TOP_REE_COUNTRIES["country"]) == {"China", "Brazil", "Australia", "Russia", "Vietnam"}
assert set(REE_ELEMENTS["symbol"]) == {"Sc", "Y", "La", "Ce", "Pr", "Nd", "Pm", "Sm", "Eu", "Gd", "Tb", "Dy", "Ho", "Er", "Tm", "Yb", "Lu"}
assert {"query_country", "query_element", "latitude", "longitude", "analysis_ready"}.issubset(clean.columns)

print("All checks passed.")
matrix


All checks passed.


symbol,Sc,Y,La,Ce,Pr,Nd,Pm,Sm,Eu,Gd,Tb,Dy,Ho,Er,Tm,Yb,Lu
country,,,,,,,,,,,,,,,,,
China,6,114,40,109,0,63,0,0,0,0,0,0,0,0,0,3,0
Brazil,0,52,17,67,0,21,0,0,0,0,0,0,0,0,0,0,0
Australia,7,150,42,111,0,38,0,0,0,0,0,0,0,0,0,0,0
Russia,8,125,62,216,0,59,0,4,0,1,0,0,0,0,0,2,0
Vietnam,0,4,4,7,0,6,0,0,0,0,0,0,0,0,0,0,0


---

# Downstream Team Workspace

Qiwei's data-engineering part ends here.

Recommended files for later members:

- `data/processed/top5_17ree_localities_analysis_ready.csv`
- `data/processed/top5_17ree_distribution_summary.csv`
- `data/processed/top5_17ree_distribution_matrix.csv`


## Mengna Cheng - GIS / Visualization

Blank workspace for maps and charts.


## Michael King - Clustering / ML

My part is figuring out whether REE deposit locations naturally group together and whether those groups have different mixes of light vs heavy rare earth elements. I use clustering to find the groups and a classifier to see if the groupings actually mean something geochemically.

**Data from Qiwei:**
- data/processed/top5_17ree_localities_analysis_ready.csv
- data/processed/top5_17ree_distribution_summary.csv
- data/processed/top5_17ree_distribution_matrix.csv




### Setup

Install any missing packages and load everything in.


In [5]:
import subprocess
subprocess.run(['pip', 'install', 'seaborn', '-q'], check=True)

from pathlib import Path
import ast
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score,
    classification_report, confusion_matrix,
    roc_auc_score
)
from sklearn.model_selection import (
    train_test_split, cross_val_score,
    GridSearchCV, StratifiedKFold
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance

PROCESSED_DIR = Path('data/processed')
FIGURES_DIR = Path('figures/michael')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

analysisready = pd.read_csv(PROCESSED_DIR / 'top5_17ree_localities_analysis_ready.csv')
summary = pd.read_csv(PROCESSED_DIR / 'top5_17ree_distribution_summary.csv')
matrix = pd.read_csv(PROCESSED_DIR / 'top5_17ree_distribution_matrix.csv', index_col=0)

print('Rows loaded:', len(analysisready))
analysisready.head(3)


Rows loaded: 1349


,reserve_rank,query_country,reserves_reo_tonnes,query_element,element_name,ree_group,locality_id,locality_name,country,latitude,longitude,analysis_ready,reported_elements,datemodify,parent,coordsystem,mindat_locality_url
0,1,China,44000000,Sc,Scandium,scandium,715,"Xianghualing Mine, Xianghualing Sn-polymetalli...",China,25.458611,112.567222,True,"['Ag', 'S', 'Ca', 'Fe', 'Mg', 'Si', 'O', 'H', ...",2026-04-20 20:33:37,157020,0,https://www.mindat.org/loc-715.html
1,1,China,44000000,Sc,Scandium,scandium,720,"Bayan Obo deposit, Bayan Obo mining district, ...",China,41.795833,109.969444,True,"['Ca', 'Fe', 'Mg', 'Si', 'O', 'H', 'Na', 'Al',...",2026-04-02 05:26:39,185580,0,https://www.mindat.org/loc-720.html
2,1,China,44000000,Sc,Scandium,scandium,156631,Ningqiang meteorite (Ningqiang carbonaceous ch...,China,32.926389,105.906667,True,"['Ca', 'Mg', 'Si', 'O', 'Al', 'Ti', 'Fe', 'Ni'...",2025-06-21 09:47:27,185604,0,https://www.mindat.org/loc-156631.html


### Building the Features

I need to turn the raw locality data into something a model can use. For each site I record its coordinates and which of the 17 REE elements were reported there. I also create a simple yes or no label for whether a site reports more light REEs than heavy ones which is what I will try to predict later.


In [6]:
LREE = {'La', 'Ce', 'Pr', 'Nd', 'Pm', 'Sm', 'Eu'}
HREE = {'Y', 'Gd', 'Tb', 'Dy', 'Ho', 'Er', 'Tm', 'Yb', 'Lu'}
ALLREE = sorted(LREE | HREE | {'Sc'})

df = analysisready.dropna(subset=['latitude', 'longitude']).copy()

def parseelements(val):
    try:
        return ast.literal_eval(val) if isinstance(val, str) else val
    except Exception:
        return []

df['elemlist'] = df['reported_elements'].apply(parseelements)

for sym in ALLREE:
    df['has' + sym] = df['elemlist'].apply(lambda e: int(sym in e))

elemcols = ['has' + sym for sym in ALLREE]

df['lreecount'] = df['elemlist'].apply(lambda e: len(LREE & set(e)))
df['hreecount'] = df['elemlist'].apply(lambda e: len(HREE & set(e)))
df['lreedominant'] = (df['lreecount'] >= df['hreecount']).astype(int)

print('Localities with coordinates:', len(df))
print('LREE dominant sites:', df['lreedominant'].sum(),
      f"({df['lreedominant'].mean()*100:.1f}%)")


Localities with coordinates: 1349
LREE dominant sites: 1187 (88.0%)


### Scaling and PCA

Before clustering I scale all the features so nothing dominates just because of its units. Latitude and longitude are in degrees but the element flags are 0s and 1s so without scaling the coordinates would drown out the element information. I also run PCA to see how much of the variation can be explained by a smaller number of dimensions.


In [7]:
featurecols = ['latitude', 'longitude'] + elemcols
Xraw = df[featurecols].fillna(0).values
scaler = StandardScaler()
Xscaled = scaler.fit_transform(Xraw)

pca = PCA(random_state=42).fit(Xscaled)
cumvar = np.cumsum(pca.explained_variance_ratio_)

n80 = int(np.searchsorted(cumvar, 0.80)) + 1
n95 = int(np.searchsorted(cumvar, 0.95)) + 1
print(f'Components needed for 80% of variance: {n80}')
print(f'Components needed for 95% of variance: {n95}')
print(f'PC1: {pca.explained_variance_ratio_[0]*100:.1f}%  PC2: {pca.explained_variance_ratio_[1]*100:.1f}%')

# Use 95% variance version for clustering to keep as much signal as possible
Xpca = PCA(n_components=n95, random_state=42).fit_transform(Xscaled)


Components needed for 80% of variance: 7
Components needed for 95% of variance: 9
PC1: 18.2%  PC2: 13.5%


### Choosing the Number of Clusters (Hypothesis 1)

To figure out how many groups to split the data into I test values of k from 2 to 10. I use three different scores to judge each option rather than just guessing from a plot so the choice is backed up by numbers.


In [9]:
krange = range(2, 11)
inertias = []
silscores = []
dbscores = []

for k in krange:
    km = KMeans(n_clusters=k, random_state=42, n_init=15, max_iter=500)
    labels = km.fit_predict(Xpca)
    inertias.append(km.inertia_)
    silscores.append(silhouette_score(Xpca, labels, sample_size=min(5000, len(Xpca))))
    dbscores.append(davies_bouldin_score(Xpca, labels))

tuningresults = pd.DataFrame({
    'k': list(krange),
    'inertia': inertias,
    'silhouette': silscores,
    'davies bouldin': dbscores,
})

bestk = tuningresults.loc[tuningresults['silhouette'].idxmax(), 'k']
print(tuningresults.to_string(index=False))
print(f'\nGoing with k = {bestk}  (best silhouette score = {tuningresults["silhouette"].max():.4f})')


 k      inertia  silhouette  davies bouldin
 2 11064.129179    0.230260        1.930845
 3  9706.411143    0.246657        1.369469
 4  8450.209538    0.276544        1.256391
 5  7100.080542    0.290884        0.987788
 6  5771.006192    0.310289        0.888506
 7  4496.063363    0.337763        0.812240
 8  3587.765656    0.371171        0.766438
 9  2887.072493    0.413537        0.699506
10  2497.545957    0.433093        0.686626

Going with k = 10  (best silhouette score = 0.4331)


### Fitting the Final Clusters

With the best k chosen I refit the model and check how the clusters break down. The key thing I am looking at is whether different clusters have different LREE vs HREE balances.


In [10]:
kmfinal = KMeans(n_clusters=int(bestk), random_state=42, n_init=20, max_iter=500)
df['kmcluster'] = kmfinal.fit_predict(Xpca)

silfinal = silhouette_score(Xpca, df['kmcluster'], sample_size=min(5000, len(Xpca)))
dbfinal = davies_bouldin_score(Xpca, df['kmcluster'])

print(f'KMeans  k={bestk}')
print(f'  Silhouette: {silfinal:.4f}  (closer to 1 is better)')
print(f'  Davies Bouldin: {dbfinal:.4f}  (closer to 0 is better)')
print()
print('Cluster sizes:')
print(df['kmcluster'].value_counts().sort_index())
print()
print('LREE vs HREE balance per cluster:')
balance = (
    df.groupby('kmcluster')[['lreecount', 'hreecount', 'lreedominant']]
    .mean()
    .rename(columns={'lreecount': 'avg lree', 'hreecount': 'avg hree', 'lreedominant': 'pct lree dominant'})
)
balance['pct lree dominant'] = (balance['pct lree dominant'] * 100).round(1)
print(balance.round(3))


KMeans  k=10
  Silhouette: 0.4331  (closer to 1 is better)
  Davies Bouldin: 0.6866  (closer to 0 is better)

Cluster sizes:
kmcluster
0    289
1    161
2    183
3    215
4     33
5    171
6     15
7      1
8    263
9     18
Name: count, dtype: int64

LREE vs HREE balance per cluster:
           avg lree  avg hree  pct lree dominant
kmcluster                                       
0             2.865     0.896              100.0
1             1.596     0.758               91.3
2             0.268     0.929               26.8
3             1.205     0.000              100.0
4             0.909     0.424               93.9
5             2.117     0.930              100.0
6             1.133     2.000               26.7
7             0.000     1.000                0.0
8             1.605     1.000              100.0
9             3.556     1.000              100.0


### Density Based Clustering

KMeans always assigns every point to a cluster even if it does not really belong anywhere. DBSCAN works differently. It finds areas with lots of nearby sites and marks isolated ones as noise. I test a few different settings to find what works best rather than just picking numbers arbitrarily.

Worth noting that the best settings ended up marking about a third of all sites as noise. That is actually an interesting result on its own since it means a lot of deposit locations are geographically isolated rather than part of a dense region.


In [11]:
# Scale coordinates to rough kilometres so the distance measure makes sense
Xgeo = df[['latitude', 'longitude']].values * 111.0

epsvalues = [300, 500, 750, 1000]
minsamplevalues = [5, 10, 20]

dbresults = []
for eps in epsvalues:
    for ms in minsamplevalues:
        labels = DBSCAN(eps=eps, min_samples=ms, algorithm='ball_tree').fit_predict(Xgeo)
        nclusters = len(set(labels)) - (1 if -1 in labels else 0)
        nnoise = (labels == -1).sum()
        sil = (
            silhouette_score(Xgeo[labels != -1], labels[labels != -1],
                             sample_size=min(3000, (labels != -1).sum()))
            if nclusters > 1 else float('nan')
        )
        dbresults.append({
            'radius km': eps, 'min sites': ms,
            'clusters found': nclusters, 'noise points': nnoise,
            'noise pct': round(nnoise / len(labels) * 100, 1),
            'silhouette': round(sil, 4) if sil == sil else float('nan')
        })

dbsummary = pd.DataFrame(dbresults)
print(dbsummary.to_string(index=False))

bestdb = dbsummary.dropna(subset=['silhouette']).sort_values('silhouette', ascending=False).iloc[0]
print(f'\nBest settings: radius={bestdb["radius km"]} km  min sites={int(bestdb["min sites"])}')
print(f'  Clusters: {int(bestdb["clusters found"])}  Noise: {int(bestdb["noise points"])} ({bestdb["noise pct"]}%)  Silhouette: {bestdb["silhouette"]}')


 radius km  min sites  clusters found  noise points  noise pct  silhouette
       300          5              44            99        7.3      0.5569
       300         10              27           233       17.3      0.6685
       300         20              15           457       33.9      0.8173
       500          5              27            59        4.4      0.4490
       500         10              19           127        9.4      0.5386
       500         20               9           291       21.6      0.6317
       750          5              13            15        1.1      0.4388
       750         10              10            38        2.8      0.5616
       750         20               5           155       11.5      0.7608
      1000          5               7            10        0.7      0.6206
      1000         10               7            20        1.5      0.6465
      1000         20               5            65        4.8      0.7398

Best settings: radius=30

In [12]:
dbmodel = DBSCAN(
    eps=int(bestdb['radius km']),
    min_samples=int(bestdb['min sites']),
    algorithm='ball_tree'
)
df['dbcluster'] = dbmodel.fit_predict(Xgeo)

print('DBSCAN cluster counts (use -1 means the site was marked as noise):')
print(df['dbcluster'].value_counts().sort_index())
print()
df['dbgroup'] = df['dbcluster'].apply(lambda x: 'noise' if x == -1 else 'cluster')
print('LREE balance: clustered sites vs isolated sites:')
print(df.groupby('dbgroup')[['lreedominant', 'lreecount', 'hreecount']].mean().round(3))


DBSCAN cluster counts (use -1 means the site was marked as noise):
dbcluster
-1     457
 0      68
 1      27
 2     101
 3      51
 4     118
 5      25
 6     107
 7      24
 8      46
 9      35
 10    143
 11     29
 12     71
 13     27
 14     20
Name: count, dtype: int64

LREE balance: clustered sites vs isolated sites:
         lreedominant  lreecount  hreecount
dbgroup                                    
cluster          0.89      1.688      0.753
noise            0.86      1.711      0.796


### Comparing Classifiers hypothesis 2

Now I want to see if I can predict whether a site is LREE or HREE dominant based on where it is and what elements appear there. I try three different models and compare them using cross validation before committing to one. All three ended up agreeing pretty closely which is reassuring.


In [13]:
clffeatures = ['latitude', 'longitude', 'kmcluster'] + elemcols
Xclf = df[clffeatures].fillna(0)
yclf = df['lreedominant']

Xtrain, Xtest, ytrain, ytest = train_test_split(
    Xclf, yclf, test_size=0.20, random_state=42, stratify=yclf
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=200, random_state=42),
}

header = f"{'Model':<25} {'CV Accuracy':>12}  {'CV F1 macro':>12}"
print(header)
print('-' * 53)
cvresults = {}
for name, model in models.items():
    acc = cross_val_score(model, Xtrain, ytrain, cv=cv, scoring='accuracy', n_jobs=-1)
    f1 = cross_val_score(model, Xtrain, ytrain, cv=cv, scoring='f1_macro', n_jobs=-1)
    cvresults[name] = {'accmean': acc.mean(), 'accstd': acc.std(), 'f1mean': f1.mean(), 'f1std': f1.std()}
    print(f'{name:<25} {acc.mean():.4f} +/- {acc.std():.4f}  {f1.mean():.4f} +/- {f1.std():.4f}')


Model                      CV Accuracy   CV F1 macro
-----------------------------------------------------
Logistic Regression       0.9981 +/- 0.0023  0.9957 +/- 0.0053
Random Forest             0.9981 +/- 0.0037  0.9955 +/- 0.0090
Gradient Boosting         0.9981 +/- 0.0037  0.9955 +/- 0.0090


### Tuning the Best Model

I take whichever model scored highest and try a range of settings for it. This is called a grid search and it just tries every combination and picks the one that performs best. I optimise for F1 rather than accuracy because the LREE and HREE classes are not perfectly balanced.


In [14]:
bestmodelname = max(cvresults, key=lambda k: cvresults[k]['f1mean'])
print(f'Best model: {bestmodelname}')

paramgrids = {
    'Logistic Regression': {
        'C': [0.01, 0.1, 1, 10],
        'penalty': ['l1', 'l2'],
        'solver': ['liblinear'],
    },
    'Random Forest': {
        'n_estimators': [100, 300],
        'max_depth': [None, 10, 20],
        'min_samples_leaf': [1, 5],
    },
    'Gradient Boosting': {
        'n_estimators': [100, 200],
        'learning_rate': [0.05, 0.1, 0.2],
        'max_depth': [3, 5],
    },
}

gridsearch = GridSearchCV(
    models[bestmodelname], paramgrids[bestmodelname],
    cv=cv, scoring='f1_macro', n_jobs=-1, verbose=1, refit=True
)
gridsearch.fit(Xtrain, ytrain)

print(f'\nBest settings: {gridsearch.best_params_}')
print(f'Best CV F1: {gridsearch.best_score_:.4f}')


Best model: Logistic Regression
Fitting 5 folds for each of 8 candidates, totalling 40 fits

Best settings: {'C': 10, 'penalty': 'l1', 'solver': 'liblinear'}
Best CV F1: 1.0000


### Results on the Test Set

I held back 20% of the data from the start so the model had never seen it. Testing on this unseen data gives a more realistic picture of how well the model actually works. Precision recall and F1 all tell slightly different parts of the story so I report all three.

the near perfect scores here are worth being upfront about. The features used to predict LREE dominance include things like whether Ce or Nd appear at a site and those are themselves LREE elements that directly determine the label. So the model is essentially picking up on a circular relationship in the data rather than learning something new. This is discussed more in the limitations below.


In [15]:
tunedmodel = gridsearch.best_estimator_
ypred = tunedmodel.predict(Xtest)
yprob = tunedmodel.predict_proba(Xtest)[:, 1]

print('=== Test Set Results ===')
print(classification_report(ytest, ypred, target_names=['HREE dominant', 'LREE dominant']))

rocauc = roc_auc_score(ytest, yprob)
print(f'ROC AUC: {rocauc:.4f}')

print('\n=== Confusion Matrix ===')
cm = confusion_matrix(ytest, ypred)
cmdf = pd.DataFrame(
    cm,
    index=['Actual HREE', 'Actual LREE'],
    columns=['Predicted HREE', 'Predicted LREE']
)
print(cmdf)


=== Test Set Results ===
               precision    recall  f1-score   support

HREE dominant       1.00      0.97      0.98        32
LREE dominant       1.00      1.00      1.00       238

     accuracy                           1.00       270
    macro avg       1.00      0.98      0.99       270
 weighted avg       1.00      1.00      1.00       270

ROC AUC: 1.0000

=== Confusion Matrix ===
             Predicted HREE  Predicted LREE
Actual HREE              31               1
Actual LREE               0             238


### Which Features Matter Most

To understand what the model is actually picking up on I shuffle each feature one at a time and measure how much the score drops. If shuffling a feature hurts performance a lot that feature is important. This is more honest than the built in importance scores that some models report.

The results confirm the leakage issue. Ce Nd and La which are all light rare earth elements sit at the top by a large margin while location features like latitude and longitude contribute nothing. This makes sense because knowing a site has Ce basically tells you it is LREE dominant by definition.


In [16]:
perm = permutation_importance(
    tunedmodel, Xtest, ytest,
    n_repeats=15, random_state=42, n_jobs=-1, scoring='f1_macro'
)

impdf = pd.DataFrame({
    'feature': clffeatures,
    'avg drop': perm.importances_mean,
    'std': perm.importances_std,
}).sort_values('avg drop', ascending=False)

print('Top 15 most important features:')
print(impdf.head(15).to_string(index=False))


Top 15 most important features:
  feature  avg drop      std
    hasCe  0.379450 0.053555
    hasNd  0.197044 0.036912
    hasLa  0.165529 0.037117
     hasY  0.093133 0.015158
    hasYb  0.012147 0.004117
    hasLu  0.000000 0.000000
 latitude  0.000000 0.000000
longitude  0.000000 0.000000
    hasGd  0.000000 0.000000
    hasEu  0.000000 0.000000
    hasDy  0.000000 0.000000
    hasEr  0.000000 0.000000
    hasSm  0.000000 0.000000
    hasHo  0.000000 0.000000
    hasPm  0.000000 0.000000


### Element Presence by Cluster

This table shows the average rate at which each REE element is reported per cluster. It is a straightforward way to see whether some clusters are dominated by certain elements which directly answers Hypothesis 2.

Looking at the results clusters 2 and 6 stand out as genuinely HREE heavy with only about 27% of sites being LREE dominant. Cluster 7 is the most extreme with 0% LREE dominant and almost entirely made up of gd occurrences. These are the clusters that give the clearest answer to hypothesis 2.


In [17]:
clusterprofile = (
    df.groupby('kmcluster')[elemcols]
    .mean()
    .rename(columns=lambda c: c.replace('has', ''))
    .round(4)
)

print('Average element presence rate per cluster:')
print(clusterprofile.to_string())

print('\nPercent LREE dominant per cluster:')
print((df.groupby('kmcluster')['lreedominant'].mean() * 100).round(1).rename('pct lree dominant'))


Average element presence rate per cluster:
               Ce   Dy   Er   Eu   Gd   Ho      La   Lu      Nd   Pm   Pr   Sc   Sm   Tb   Tm       Y   Yb
kmcluster                                                                                                 
0          0.9896  0.0  0.0  0.0  0.0  0.0  1.0000  0.0  0.8754  0.0  0.0  0.0  0.0  0.0  0.0  0.8962  0.0
1          0.8696  0.0  0.0  0.0  0.0  0.0  0.3043  0.0  0.4224  0.0  0.0  0.0  0.0  0.0  0.0  0.7578  0.0
2          0.0000  0.0  0.0  0.0  0.0  0.0  0.2350  0.0  0.0328  0.0  0.0  0.0  0.0  0.0  0.0  0.9290  0.0
3          0.9860  0.0  0.0  0.0  0.0  0.0  0.2047  0.0  0.0140  0.0  0.0  0.0  0.0  0.0  0.0  0.0000  0.0
4          0.4848  0.0  0.0  0.0  0.0  0.0  0.1515  0.0  0.2727  0.0  0.0  1.0  0.0  0.0  0.0  0.4242  0.0
5          1.0000  0.0  0.0  0.0  0.0  0.0  0.4269  0.0  0.6901  0.0  0.0  0.0  0.0  0.0  0.0  0.9298  0.0
6          0.8667  0.0  0.0  0.0  0.0  0.0  0.2667  0.0  0.0000  0.0  0.0  0.0  0.0  0.0  0.0  1.0000

### Saving Outputs


In [18]:
mlout = PROCESSED_DIR / 'top5_17ree_localities_ml.csv'
profileout = PROCESSED_DIR / 'cluster_element_profile.csv'
dbout = PROCESSED_DIR / 'top5_17ree_localities_dbscan.csv'

df.drop(columns=['elemlist'], errors='ignore').to_csv(mlout, index=False)
clusterprofile.to_csv(profileout)
df[['locality_id', 'latitude', 'longitude', 'dbcluster', 'lreedominant']].to_csv(dbout, index=False)

print('Saved:', mlout)
print('Saved:', profileout)
print('Saved:', dbout)


Saved: data\processed\top5_17ree_localities_ml.csv
Saved: data\processed\cluster_element_profile.csv
Saved: data\processed\top5_17ree_localities_dbscan.csv


### Summary

**Hypothesis 1: Do REE deposits cluster spatially?**
Yes. Both KMeans and DBSCAN find consistent groupings in the data. The fact that two very different methods agree suggests the clusters are real rather than just an artefact of one algorithm. DBSCAN also found that about a third of all sites are too geographically isolated to belong to any dense group which is itself an interesting finding.

**Hypothesis 2: Do clusters differ in LREE vs HREE composition?**
Yes. The cluster balance table shows real differences in LREE and HREE proportions across groups. Clusters 2 and 6 are around 27% LREE dominant and cluster 7 is almost entirely HREE with close to 0% LREE sites. This gives a clear answer to Hypothesis 2 even without the classifier.

**Limitations:**
- The data is reporting counts not actual concentrations so clusters reflect where elements get documented rather than where they are most abundant.
- The classifier scores look perfect but that is because the features used to predict LREE dominance include the individual element flags like Ce and Nd which are themselves light rare earth elements. The model is essentially picking up on a circular relationship rather than learning something genuinely predictive. The clustering results are more trustworthy for answering Hypothesis 2.
- DBSCAN uses a flat distance approximation which works well enough at a country scale but is less accurate for global comparisons.


## Rohit Narwal - Technical Writing

Blank workspace for report notes and tables.


## Sources

- Local PDF: `DS _Assignment4a_Qiwei_Liang.pdf`
- Mindat API schema: `https://api.mindat.org/v1/schema/`
- USGS Mineral Commodity Summaries 2026, Rare Earths chapter
